API Middleware Simulator

In [26]:
#Imports
import logging
import time
from functools import wraps

In [27]:
#Logging Configuration
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger("API_Middleware")

In [28]:
#Timing Decorator
def timing(func):
    
    @wraps(func)
    def wrapper(*args, **kwargs):
        
        start_time = time.time()
        
        result = func(*args, **kwargs)
        
        end_time = time.time()
        
        logger.info(f"{func.__name__} executed in {end_time - start_time:.4f} seconds")
        
        return result
    
    return wrapper

In [29]:
#IO Logging Decorator
def log_io(func):
    
    @wraps(func)
    def wrapper(*args, **kwargs):
        
        logger.info(f"Calling {func.__name__} with args={args} kwargs={kwargs}")
        
        result = func(*args, **kwargs)
        
        logger.info(f"{func.__name__} returned {result}")
        
        return result
    
    return wrapper

In [30]:
#Access Control Decorator
def require_role(required_role):
    
    def decorator(func):
        
        @wraps(func)
        def wrapper(*args, **kwargs):
            
            user_role = kwargs.get("role", "guest")
            
            if user_role != required_role:
                
                logger.warning(
                    f"Access denied for role '{user_role}' to function {func.__name__}"
                )
                
                return "Access Denied"
            
            return func(*args, **kwargs)
        
        return wrapper
    
    return decorator

In [31]:
#API Functions
@timing
@log_io
@require_role("admin")
def get_user_profile(user_id, role="guest"):
    
    time.sleep(1)
    
    return {
        "user_id": user_id,
        "name": "Alice",
        "status": "Active"
    }

In [24]:
@timing
@log_io
@require_role("editor")
def update_article(article_id, content, role="guest"):
    
    time.sleep(0.5)
    
    return f"Article {article_id} updated successfully"

#Testing

In [25]:
# Allowed access
get_user_profile(101, role="admin")

# Access denied
get_user_profile(102, role="guest")

# Editor allowed
update_article(201, "New Content", role="editor")

# Editor denied
update_article(202, "Content", role="guest")

2026-03-08 18:49:06,598 - INFO - Calling get_user_profile with args=(101,) kwargs={'role': 'admin'}
2026-03-08 18:49:07,600 - INFO - get_user_profile returned {'user_id': 101, 'name': 'Alice', 'status': 'Active'}
2026-03-08 18:49:07,602 - INFO - get_user_profile executed in 1.0037 seconds
2026-03-08 18:49:07,603 - INFO - Calling get_user_profile with args=(102,) kwargs={'role': 'guest'}
2026-03-08 18:49:07,604 - WARNING - Access denied for role 'guest' to function get_user_profile
2026-03-08 18:49:07,604 - INFO - get_user_profile returned Access Denied
2026-03-08 18:49:07,605 - INFO - get_user_profile executed in 0.0024 seconds
2026-03-08 18:49:07,606 - INFO - Calling update_article with args=(201, 'New Content') kwargs={'role': 'editor'}
2026-03-08 18:49:08,115 - INFO - update_article returned Article 201 updated successfully
2026-03-08 18:49:08,175 - INFO - update_article executed in 0.5686 seconds
2026-03-08 18:49:08,192 - INFO - Calling update_article with args=(202, 'Content') kwa

'Access Denied'